# Behavioral Patterns — Advanced

## What's covered

- **State** — let an object change behaviour when its internal state changes
- **Chain of Responsibility** — pass a request along a chain until one handler accepts it
- **Mediator** — centralize how a set of objects communicate
- **Memento** — capture an object's state so it can be restored later
- **Visitor** — add operations to a class hierarchy without modifying the classes
- How each pattern shows up in practice and when to reach for it


## The remaining five behavioral patterns

These five show up less often than the core five from notebook 04, but each solves a specific problem cleanly when its shape matches yours.

- **State** is for objects with a clear lifecycle — order, connection, document workflow — where behaviour changes with mode.
- **Chain of Responsibility** is for layered processing — middleware, validation pipelines, approval workflows.
- **Mediator** is for many-to-many coordination — UI dialogs, chat rooms — that would otherwise tangle into a mesh of direct references.
- **Memento** is for snapshot-based undo and rollback — the alternative to Command-based undo when you'd rather capture state than reverse operations.
- **Visitor** is for adding new operations to a closed class hierarchy without modifying the classes themselves. In modern languages with sealed types and pattern matching, this collapses to a `when` / `match` expression.

A useful frame: each of these patterns sacrifices simplicity for a specific property — explicit state transitions, layered processing, decoupled communication, captured state, externally-extendable operations. If you don't need the property, you don't need the pattern.


## State

**Intent:** allow an object to alter its behaviour when its internal state changes. The object will appear to change its class.

**Where it shows up:** order lifecycles (draft → placed → paid → shipped → delivered), connection states (disconnected → connecting → open → closing → closed), document workflows (draft → review → published → archived), game character modes (idle / running / jumping / dying), traffic light controllers.

**The shape:** the context holds a reference to a state object. Each state class implements the same interface but handles methods differently — and crucially, each state controls *which states the object can transition to next*. The context delegates work to the current state, which may also tell the context to change to a new state.

**The win over a giant `if/elif on self.status`:** state transitions become explicit, and invalid transitions become impossible. A "paid" order knows that `ship()` is valid but `cancel()` requires a refund flow; the type system or the method dispatch enforces this. A status string in a giant `if` ladder enforces nothing.

**Python's take:** a small class per state, all implementing a common protocol. The context holds the current state instance.

**Kotlin's take:** a `sealed interface` with one object/class per state, plus a `when` expression for transitions. Kotlin's sealed-type exhaustiveness check catches missed states at compile time.


### Python


In [ ]:
from typing import Protocol


class OrderState(Protocol):
    def pay(self, order: "Order") -> None: ...
    def ship(self, order: "Order") -> None: ...
    def cancel(self, order: "Order") -> None: ...
    def name(self) -> str: ...


class Order:
    def __init__(self):
        self._state: OrderState = Draft()

    def transition_to(self, state: OrderState) -> None:
        print(f"  {self._state.name()} -> {state.name()}")
        self._state = state

    def pay(self):    self._state.pay(self)
    def ship(self):   self._state.ship(self)
    def cancel(self): self._state.cancel(self)


def _illegal(action: str, state_name: str) -> None:
    raise RuntimeError(f"cannot {action} a {state_name} order")


class Draft:
    def name(self) -> str: return "Draft"
    def pay(self, order: Order):    order.transition_to(Paid())
    def ship(self, order: Order):   _illegal("ship", "Draft")
    def cancel(self, order: Order): order.transition_to(Cancelled())


class Paid:
    def name(self) -> str: return "Paid"
    def pay(self, order: Order):    _illegal("pay", "Paid")
    def ship(self, order: Order):   order.transition_to(Shipped())
    def cancel(self, order: Order): order.transition_to(Refunding())


class Shipped:
    def name(self) -> str: return "Shipped"
    def pay(self, order: Order):    _illegal("pay", "Shipped")
    def ship(self, order: Order):   _illegal("ship", "Shipped")
    def cancel(self, order: Order): _illegal("cancel", "Shipped")


class Refunding:
    def name(self) -> str: return "Refunding"
    def pay(self, order: Order):    _illegal("pay", "Refunding")
    def ship(self, order: Order):   _illegal("ship", "Refunding")
    def cancel(self, order: Order): _illegal("cancel", "Refunding")


class Cancelled:
    def name(self) -> str: return "Cancelled"
    def pay(self, order: Order):    _illegal("pay", "Cancelled")
    def ship(self, order: Order):   _illegal("ship", "Cancelled")
    def cancel(self, order: Order): _illegal("cancel", "Cancelled")


# Walk a successful order through the happy path.
order = Order()
order.pay()
order.ship()

# Try an illegal transition.
try:
    order.cancel()
except RuntimeError as e:
    print(f"  blocked: {e}")


### Kotlin

```kotlin
sealed interface OrderState {
    val name: String
}
data object Draft     : OrderState { override val name = "Draft" }
data object Paid      : OrderState { override val name = "Paid" }
data object Shipped   : OrderState { override val name = "Shipped" }
data object Refunding : OrderState { override val name = "Refunding" }
data object Cancelled : OrderState { override val name = "Cancelled" }

class Order {
    var state: OrderState = Draft
        private set

    private fun transitionTo(next: OrderState) {
        println("  ${state.name} -> ${next.name}")
        state = next
    }

    fun pay() = when (state) {
        Draft -> transitionTo(Paid)
        else  -> error("cannot pay a ${state.name} order")
    }

    fun ship() = when (state) {
        Paid -> transitionTo(Shipped)
        else -> error("cannot ship a ${state.name} order")
    }

    fun cancel() = when (state) {
        Draft -> transitionTo(Cancelled)
        Paid  -> transitionTo(Refunding)
        else  -> error("cannot cancel a ${state.name} order")
    }
}

val order = Order()
order.pay()
order.ship()
try { order.cancel() } catch (e: IllegalStateException) { println("  blocked: ${e.message}") }
```


**When not to use State.** When the object has only two or three states and the transition rules are simple, a status enum plus a small `if` chain is clearer than five state classes. State earns its keep when transition rules are intricate, when invalid transitions need to be impossible (not just wrong), and when each state has substantial behaviour of its own.


## Chain of Responsibility

**Intent:** avoid coupling the sender of a request to its receiver by giving more than one object a chance to handle it. Chain the receiving objects and pass the request along the chain until an object handles it.

**Where it shows up:** HTTP middleware (auth → rate limit → logging → handler), event bubbling in DOM and UI frameworks, validation pipelines, approval workflows (junior → manager → director), exception handlers, logging frameworks (handler-chain on a logger).

**The shape:** each handler has a reference to the next handler in the chain. When a request arrives, the handler either processes it and returns, or forwards it to the next handler. The sender just hands the request to the head of the chain.

**Two flavours:**

- **Pass-through** — every handler runs, each adding its own contribution before forwarding (typical for middleware).
- **First-handler-wins** — handlers check whether they can handle the request; the first that can does, and the rest are skipped (typical for approval workflows).

**Python's take:** a list of callables iterated until one returns a result, or a linked list of handler objects each with a `next` reference.

**Kotlin's take:** the same. Coroutine `Flow` operators give you a more declarative pass-through chain when ordering matters.


### Python


In [ ]:
from dataclasses import dataclass


@dataclass
class Request:
    user: str
    role: str
    amount: int


# First-handler-wins flavour — approval chain by amount thresholds.
class Approver:
    def __init__(self, name: str, limit: int):
        self.name = name
        self.limit = limit
        self._next: "Approver | None" = None

    def chain(self, next_: "Approver") -> "Approver":
        self._next = next_
        return next_

    def handle(self, req: Request) -> str:
        if req.amount <= self.limit:
            return f"{self.name} approved ${req.amount} for {req.user}"
        if self._next is None:
            return f"no one in chain can approve ${req.amount}"
        return self._next.handle(req)


# Build the chain: team lead -> manager -> director
team_lead = Approver("TeamLead", 1_000)
manager   = Approver("Manager",  10_000)
director  = Approver("Director", 100_000)
team_lead.chain(manager).chain(director)

print(team_lead.handle(Request("alice", "engineer", 500)))
print(team_lead.handle(Request("bob",   "engineer", 5_000)))
print(team_lead.handle(Request("carol", "manager",  75_000)))
print(team_lead.handle(Request("dave",  "director", 1_000_000)))


# Pass-through flavour — middleware chain.
from typing import Callable

Handler = Callable[[Request], str]
Middleware = Callable[[Request, Handler], str]


def with_auth(req: Request, next_: Handler) -> str:
    if not req.role:
        return "denied: no role"
    return next_(req)


def with_logging(req: Request, next_: Handler) -> str:
    result = next_(req)
    print(f"  [log] {req.user}: {result}")
    return result


def final_handler(req: Request) -> str:
    return f"ok ${req.amount}"


def chain_middleware(mws: list[Middleware], final: Handler) -> Handler:
    handler = final
    for mw in reversed(mws):
        prev = handler
        handler = lambda r, mw=mw, p=prev: mw(r, p)
    return handler


pipeline = chain_middleware([with_logging, with_auth], final_handler)
print(pipeline(Request("alice", "engineer", 42)))
print(pipeline(Request("eve", "", 42)))


### Kotlin

```kotlin
data class Request(val user: String, val role: String, val amount: Int)

class Approver(val name: String, private val limit: Int) {
    private var next: Approver? = null
    fun chain(n: Approver): Approver { next = n; return n }
    fun handle(req: Request): String = when {
        req.amount <= limit -> "$name approved $${req.amount} for ${req.user}"
        next != null        -> next!!.handle(req)
        else                -> "no one in chain can approve $${req.amount}"
    }
}

val teamLead = Approver("TeamLead", 1_000)
val manager  = Approver("Manager", 10_000)
val director = Approver("Director", 100_000)
teamLead.chain(manager).chain(director)

println(teamLead.handle(Request("alice", "engineer", 500)))
println(teamLead.handle(Request("bob", "engineer", 5_000)))
println(teamLead.handle(Request("carol", "manager", 75_000)))
println(teamLead.handle(Request("dave", "director", 1_000_000)))

// Pass-through middleware chain
typealias Handler = (Request) -> String
typealias Middleware = (Request, Handler) -> String

fun chain(mws: List<Middleware>, final: Handler): Handler =
    mws.foldRight(final) { mw, next -> { req -> mw(req, next) } }

val withAuth: Middleware = { req, next -> if (req.role.isEmpty()) "denied" else next(req) }
val withLog: Middleware = { req, next ->
    val result = next(req)
    println("  [log] ${req.user}: $result")
    result
}

val pipeline = chain(listOf(withLog, withAuth)) { "ok $${it.amount}" }
println(pipeline(Request("alice", "engineer", 42)))
```


**When not to use Chain of Responsibility.** When you know exactly which handler should run, just call it. The pattern adds value when the choice of handler is dynamic, when handlers can be plugged in independently (middleware!), or when the chain itself is the configuration (auth pipeline, validation rules). The hidden cost: a request that no one handles can silently disappear unless you add a terminal handler. Always include one.


## Mediator

**Intent:** define an object that encapsulates how a set of objects interact. Mediator promotes loose coupling by keeping objects from referring to each other explicitly; it lets you vary their interaction independently.

**Where it shows up:** UI dialog forms (changing one field updates several others), chat rooms (users send to a room, the room broadcasts), air traffic control (aircraft don't talk to each other, they all talk to ATC), event-bus libraries, message brokers.

**The shape:** participants (colleagues) hold a reference to the mediator but not to each other. When a colleague wants to communicate or react to something, it tells the mediator. The mediator coordinates the actual interaction — calling the right method on the right colleague.

**The win:** in a system with N colleagues that all might react to each other, the naïve mesh has N² connections. With a mediator, each colleague has one connection (to the mediator); the mediator owns the N² relationship logic in one place. Adding a new colleague means updating one class instead of N.

**The cost:** the mediator becomes a god class as the system grows. If you're noticing the mediator has thousands of lines, the coordination logic probably needs to be split — perhaps into multiple mediators by topic.

**The contrast with Observer.** Observer is *one-to-many broadcast* — the subject doesn't care who's listening. Mediator is *coordinated many-to-many* — the mediator knows the colleagues and decides who needs what.


### Python


In [ ]:
from typing import Protocol


class ChatRoom(Protocol):
    def join(self, user: "User") -> None: ...
    def broadcast(self, sender: "User", message: str) -> None: ...
    def whisper(self, sender: "User", recipient: str, message: str) -> None: ...


class User:
    def __init__(self, name: str, room: ChatRoom):
        self.name = name
        self._room = room
        room.join(self)

    def send(self, message: str) -> None:
        self._room.broadcast(self, message)

    def whisper(self, recipient: str, message: str) -> None:
        self._room.whisper(self, recipient, message)

    def receive(self, sender_name: str, message: str, private: bool = False) -> None:
        tag = " (whisper)" if private else ""
        print(f"  [{self.name}] from {sender_name}{tag}: {message}")


class BasicChatRoom:
    def __init__(self):
        self._users: dict[str, User] = {}

    def join(self, user: User) -> None:
        self._users[user.name] = user

    def broadcast(self, sender: User, message: str) -> None:
        for name, u in self._users.items():
            if name != sender.name:
                u.receive(sender.name, message)

    def whisper(self, sender: User, recipient: str, message: str) -> None:
        if recipient in self._users:
            self._users[recipient].receive(sender.name, message, private=True)


room = BasicChatRoom()
alice = User("Alice", room)
bob   = User("Bob",   room)
carol = User("Carol", room)

alice.send("hello everyone")        # bob and carol receive
bob.whisper("Alice", "psst, want coffee?")  # only alice receives


### Kotlin

```kotlin
interface ChatRoom {
    fun join(user: User)
    fun broadcast(sender: User, message: String)
    fun whisper(sender: User, recipient: String, message: String)
}

class User(val name: String, private val room: ChatRoom) {
    init { room.join(this) }
    fun send(message: String) = room.broadcast(this, message)
    fun whisper(recipient: String, message: String) = room.whisper(this, recipient, message)
    fun receive(senderName: String, message: String, private: Boolean = false) {
        val tag = if (private) " (whisper)" else ""
        println("  [$name] from $senderName$tag: $message")
    }
}

class BasicChatRoom : ChatRoom {
    private val users = mutableMapOf<String, User>()
    override fun join(user: User) { users[user.name] = user }
    override fun broadcast(sender: User, message: String) {
        users.filterKeys { it != sender.name }.values.forEach {
            it.receive(sender.name, message)
        }
    }
    override fun whisper(sender: User, recipient: String, message: String) {
        users[recipient]?.receive(sender.name, message, private = true)
    }
}

val room = BasicChatRoom()
val alice = User("Alice", room)
val bob   = User("Bob", room)
val carol = User("Carol", room)
alice.send("hello everyone")
bob.whisper("Alice", "psst, want coffee?")
```


**When not to use Mediator.** When the interaction is one-to-many broadcast (use Observer), or when there are only two colleagues (just let them talk directly). Mediator earns its keep when several colleagues interact in patterns that would otherwise tangle into a mesh of direct references. Watch the mediator's size — if it grows past a few hundred lines, consider splitting the coordination by topic before the mediator becomes a god class.


## Memento

**Intent:** without violating encapsulation, capture and externalize an object's internal state so the object can be restored to this state later.

**Where it shows up:** undo/redo (the snapshot-based alternative to Command-based undo), save points in games, document checkpoints, database transaction rollback, time-travel debugging, form-state restoration.

**The shape:** the *originator* (the object whose state we want to save) creates a *memento* — an opaque snapshot of its internal state. A *caretaker* (often a history stack) holds the mementos but does not inspect them. To restore, the caretaker hands a memento back to the originator, which uses it to reset itself.

**Memento versus Command-based undo:**

- **Command** records the *operations*; undo means *reversing* each operation. Cheap for small operations, expensive for sweeping changes.
- **Memento** records the *state*; undo means *restoring* a snapshot. Cheap to apply, expensive in memory if snapshots are large.

Pick Command when operations are localized (text insertions, single-field edits). Pick Memento when changes are sweeping and snapshots are small (a form's field map, a game's player state). Many systems combine both.

**Python's take:** a snapshot via `copy.deepcopy` for mutable state, or via dataclass/`replace` for immutable state. Either way the originator creates the memento, the caretaker holds it.

**Kotlin's take:** an immutable data class as the memento — `.copy()` snapshots state, restoring is just reassigning a field. The pattern is largely invisible at the call site.


### Python


In [ ]:
from dataclasses import dataclass, field, replace


@dataclass(frozen=True)
class EditorMemento:
    # opaque to outsiders — only the editor knows what to do with it
    text: str
    cursor: int
    selection: tuple[int, int] | None


class Editor:
    def __init__(self):
        self.text = ""
        self.cursor = 0
        self.selection: tuple[int, int] | None = None

    def type(self, content: str) -> None:
        self.text = self.text[:self.cursor] + content + self.text[self.cursor:]
        self.cursor += len(content)

    def select(self, start: int, end: int) -> None:
        self.selection = (start, end)

    # The "memento" methods — internal API the caretaker uses.
    def snapshot(self) -> EditorMemento:
        return EditorMemento(self.text, self.cursor, self.selection)

    def restore(self, memento: EditorMemento) -> None:
        self.text = memento.text
        self.cursor = memento.cursor
        self.selection = memento.selection

    def __repr__(self) -> str:
        return f"Editor(text={self.text!r}, cursor={self.cursor}, selection={self.selection})"


class History:
    def __init__(self, editor: Editor):
        self._editor = editor
        self._stack: list[EditorMemento] = []

    def save(self) -> None:
        self._stack.append(self._editor.snapshot())

    def undo(self) -> None:
        if self._stack:
            self._editor.restore(self._stack.pop())


editor = Editor()
history = History(editor)

history.save()
editor.type("Hello")
print(editor)                 # Hello

history.save()
editor.type(", world!")
editor.select(7, 12)
print(editor)                 # Hello, world!, selection (7, 12)

history.undo()                # back to "Hello", no selection
print(editor)
history.undo()                # back to ""
print(editor)


### Kotlin

```kotlin
data class EditorMemento(val text: String, val cursor: Int, val selection: Pair<Int, Int>?)

class Editor {
    var text: String = ""; private set
    var cursor: Int = 0;   private set
    var selection: Pair<Int, Int>? = null; private set

    fun type(content: String) {
        text = text.substring(0, cursor) + content + text.substring(cursor)
        cursor += content.length
    }

    fun select(start: Int, end: Int) { selection = start to end }

    fun snapshot() = EditorMemento(text, cursor, selection)
    fun restore(m: EditorMemento) {
        text = m.text; cursor = m.cursor; selection = m.selection
    }

    override fun toString() = "Editor(text=$text, cursor=$cursor, selection=$selection)"
}

class History(private val editor: Editor) {
    private val stack = ArrayDeque<EditorMemento>()
    fun save() { stack.addLast(editor.snapshot()) }
    fun undo() { if (stack.isNotEmpty()) editor.restore(stack.removeLast()) }
}

val editor = Editor()
val history = History(editor)
history.save()
editor.type("Hello")
println(editor)
history.save()
editor.type(", world!")
editor.select(7, 12)
println(editor)
history.undo(); println(editor)
history.undo(); println(editor)
```


**When not to use Memento.** When the state is huge — a buffer of millions of objects — snapshotting after every operation will exhaust memory fast. Either move to Command-based undo (record only the diff) or use a copy-on-write data structure that shares unchanged subtrees between snapshots. Memento is also overkill when the object has only one or two fields you'd want to restore — just save them directly.


## Visitor

**Intent:** represent an operation to be performed on the elements of an object structure. Visitor lets you define a new operation without changing the classes of the elements on which it operates.

**Where it shows up:** abstract syntax tree (AST) traversal — type checking, pretty printing, evaluation, compilation are each separate visitors. Document processing (a single document tree with separate visitors for h-t-m-l export, PDF export, word-count). Game scene graphs.

**The classical motivation:** you have a closed class hierarchy (the node types) and an open set of operations (every new analysis is a new operation). Adding a new operation by adding methods to every node class is intrusive — every operation touches every class. Visitor flips it: each *operation* is one class with one method per node type. Adding a new operation is one new class; adding a new node type forces every visitor to handle it.

**The trade-off:**

- **Methods on the node classes.** Adding new operations is hard; adding new node types is easy.
- **Visitor.** Adding new operations is easy; adding new node types is hard.

Pick based on which axis is more likely to change.

**Python's take:** a visitor class with `visit_NodeType` methods, dispatched on the node's class. `functools.singledispatch` gives you the same shape with less boilerplate.

**Kotlin's take:** a `sealed class` for the node hierarchy plus a `when` expression on the node type. The compiler enforces exhaustiveness — add a new node type and every `when` expression becomes a compile error until you handle it. This is Kotlin's idiomatic answer and arguably better than the classical Visitor structure.


### Python


In [ ]:
from dataclasses import dataclass
from functools import singledispatch


# AST node hierarchy — closed set of types.
@dataclass
class Num:
    value: int


@dataclass
class Add:
    left: "Expr"
    right: "Expr"


@dataclass
class Mul:
    left: "Expr"
    right: "Expr"


Expr = Num | Add | Mul


# Visitor 1 — evaluate the expression.
@singledispatch
def evaluate(node) -> int:
    raise NotImplementedError(f"no visitor for {type(node).__name__}")


@evaluate.register
def _(node: Num) -> int:
    return node.value


@evaluate.register
def _(node: Add) -> int:
    return evaluate(node.left) + evaluate(node.right)


@evaluate.register
def _(node: Mul) -> int:
    return evaluate(node.left) * evaluate(node.right)


# Visitor 2 — pretty-print the expression. Adding this is a new module-level
# function; the node classes do not change.
@singledispatch
def pretty(node) -> str:
    raise NotImplementedError


@pretty.register
def _(node: Num) -> str:
    return str(node.value)


@pretty.register
def _(node: Add) -> str:
    return f"({pretty(node.left)} + {pretty(node.right)})"


@pretty.register
def _(node: Mul) -> str:
    return f"({pretty(node.left)} * {pretty(node.right)})"


# (3 + 4) * 5
expr: Expr = Mul(Add(Num(3), Num(4)), Num(5))
print(pretty(expr))      # ((3 + 4) * 5)
print(evaluate(expr))    # 35


### Kotlin

```kotlin
sealed interface Expr
data class Num(val value: Int) : Expr
data class Add(val left: Expr, val right: Expr) : Expr
data class Mul(val left: Expr, val right: Expr) : Expr

// Visitor 1 — evaluate. The `when` exhaustiveness check guarantees we
// handle every Expr variant; adding a new one is a compile error here.
fun evaluate(node: Expr): Int = when (node) {
    is Num -> node.value
    is Add -> evaluate(node.left) + evaluate(node.right)
    is Mul -> evaluate(node.left) * evaluate(node.right)
}

fun pretty(node: Expr): String = when (node) {
    is Num -> node.value.toString()
    is Add -> "(${pretty(node.left)} + ${pretty(node.right)})"
    is Mul -> "(${pretty(node.left)} * ${pretty(node.right)})"
}

val expr: Expr = Mul(Add(Num(3), Num(4)), Num(5))
println(pretty(expr))      // ((3 + 4) * 5)
println(evaluate(expr))    // 35
```


**When not to use Visitor.** When the node hierarchy is *not* closed — every new node type forces you to update every existing visitor. Visitor only earns its keep when the set of types is stable and the set of operations is growing. In modern languages with sealed types and pattern matching (Kotlin's `sealed` + `when`, Python's `match`, Rust's `enum` + `match`, Scala's case classes), the structural ceremony of a classical Visitor disappears — you get the same exhaustiveness guarantee from the compiler without the double-dispatch boilerplate.


## Picking the right advanced behavioral pattern

| Question | Pattern |
|---|---|
| "My object has a clear lifecycle with distinct modes and rules about transitions." | **State** |
| "I have a layered pipeline of handlers, and either every layer runs or the first match wins." | **Chain of Responsibility** |
| "Many objects need to coordinate, and direct connections would form a mesh." | **Mediator** |
| "I need snapshot-based undo or rollback." | **Memento** |
| "My class hierarchy is closed, but I keep adding new operations on it." | **Visitor** |

The notable language collapses for this group:

- **Visitor** dissolves into `sealed` types plus `when`/`match`/pattern matching in modern languages. The intent still matters; the double-dispatch boilerplate goes away.
- **Memento** dissolves into immutable data classes with a `.copy()` method. The snapshot is the data class itself; the caretaker is a list.
- **State**, **Mediator**, and **Chain of Responsibility** still benefit from explicit structure even in modern languages — the patterns name something the language doesn't already give you.


Notebook six turns to **Dependency Injection & Enterprise Patterns** — the patterns that grew up around modern service-oriented code. Dependency Injection and inversion of control as the foundation, then Repository, Unit of Work, Specification, and Null Object as the recipes that show up again and again in a typical service layer.
